In [8]:
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')


In [9]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')   

In [10]:
x_train = train_df.drop(columns=['Activity'])
y_train = train_df['Activity']
x_test = test_df.drop(columns=['Activity'])
y_test = test_df['Activity']

In [11]:
print('Calculating corealtion matrix')
corr_matrix =  x_train.corr().abs()

print('selecting upper triangle')
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))

print('removing highly corelated columns ')
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]

final_x_train = x_train.drop(columns=to_drop)

final_x_test = x_test.drop(columns=to_drop)

Calculating corealtion matrix
selecting upper triangle
removing highly corelated columns 


In [12]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), final_x_train.columns)
])

In [13]:
models = {
    'RandomForestClassifier': RandomForestClassifier(),
    'ExtraTreesClassifier': ExtraTreesClassifier(), 
}
parameters = {
    'RandomForestClassifier': {
        
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    },
    'ExtraTreesClassifier': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    }       
}

In [14]:
from sklearn.pipeline import Pipeline

for model_name, model in models.items():
    print(f"Training {model_name}...")
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    param_grid = parameters[model_name]
    
    grid_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=8, cv=5, scoring='f1_macro', n_jobs=-1)
    grid_search.fit(final_x_train, y_train)
    
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    
    print(f"Best parameters for {model_name}: {best_params}")
    print(f"Best cross-validation accuracy for {model_name}: {best_score:.4f}")
    
    test_accuracy = best_model.score(final_x_test, y_test)
    print(f"Test accuracy for {model_name}: {test_accuracy:.4f}\n")

Training RandomForestClassifier...


ValueError: Invalid parameter 'n_estimators' for estimator Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('scaler', StandardScaler(),
                                                  Index(['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z',
       'tBodyAcc-std()-X', 'tBodyAcc-entropy()-X', 'tBodyAcc-entropy()-Z',
       'tBodyAcc-arCoeff()-X,1', 'tBodyAcc-arCoeff()-X,3',
       'tBodyAcc-arCoeff()-X,4', 'tBodyAcc-arCoeff()-Y,1',
       ...
       'fBodyBodyGyroMag-skewne...fBodyBodyGyroJerkMag-min()',
       'fBodyBodyGyroJerkMag-maxInds', 'fBodyBodyGyroJerkMag-meanFreq()',
       'fBodyBodyGyroJerkMag-skewness()', 'angle(tBodyAccMean,gravity)',
       'angle(tBodyAccJerkMean),gravityMean)',
       'angle(tBodyGyroMean,gravityMean)',
       'angle(tBodyGyroJerkMean,gravityMean)', 'subject'],
      dtype='str', length=173))])),
                ('classifier', RandomForestClassifier())]). Valid parameters are: ['memory', 'steps', 'transfor